# Introduction to Napari

This notebook gives a concise, hands-on overview of how to use Napari as a scientific image viewer **and** as a programmable analysis environment. We will progressively cover:

1. **Adding data to the viewer**: images, labels, points, and shapes.
2. **Customizing viewer state**: camera, axes, grid, and other global UI controls.
3. **Programmatically altering layers**: modifying data, contrast limits, scaling, and label coloring.
4. **Building a simple widget with `magicgui`**: how type annotations with Napari types let the viewer build a proper UI.

The examples are intentionally compact and use a built‑in sample image so the notebook runs anywhere.

## Install requirements

Install the Python dependencies from the repository requirements file before running the notebook:
(make sure to be in the same folder as the "requirements.txt" file)
```bash
pip install -r requirements.txt
```

In [ ]:
# Alternatively, you could uncomment the line below and run the cell
#!pip install -r requirements.txt

In [ ]:
# Core imports for the tutorial
import napari
import numpy as np
from skimage import filters, measure
from skimage.io import imread

In [ ]:
# Launch a Napari Viewer
viewer = napari.Viewer()

## Adding data to the viewer

Napari is built around **layers**. Each call to `viewer.add_*` creates a new layer that you can toggle, style, and update independently. We'll add an image, create a labels layer from a threshold, then add points and shapes derived from region properties.

In [ ]:
# Load a sample image
image = imread(r"nuclei.tif")[41] #UPDATE WITH YOUR PATH TO THE IMAGE
image.shape

In [ ]:
# Add the image to Napari
image_layer = viewer.add_image(image, name="nuclei", colormap="gray")

In [ ]:
# Add a labels layer (simple threshold + connected components)
threshold = filters.threshold_otsu(image)
image = filters.gaussian(image, sigma=1, preserve_range=True)
mask = image > threshold
label_image = measure.label(mask)
labels_layer = viewer.add_labels(label_image, name="nuclei_labels")

In [ ]:
# Add a Points layer (centroids from region properties)
props = measure.regionprops_table(
    label_image,
    intensity_image=image,
    properties=["centroid", "area", "bbox"],
)
points = np.column_stack((props["centroid-0"], props["centroid-1"]))
points_layer = viewer.add_points(
    points,
    name="centroids",
    size=6,
    face_color="#00ffff55",
)

In [ ]:
# Add Shapes (bounding boxes)
bboxes = np.column_stack(
    (props["bbox-0"], props["bbox-1"], props["bbox-2"], props["bbox-3"])
 )
n_boxes = len(bboxes)
bounding_boxes = bboxes.reshape((n_boxes, 2, 2))
shapes_layer = viewer.add_shapes(
    data=bounding_boxes,
    shape_type="rectangle",
    edge_color="red",
    edge_width=1,
    face_color="transparent",
    name="Bounding boxes",
)

## Customizing the viewer state

Once data is in the viewer, you can programmatically change *global* viewer settings. This is useful for reproducible screenshots, predefined UI layouts, or building interactive apps. Below are some common options: camera, axes, grid view, and scale bar.

In [ ]:
# Viewer-level settings: camera
viewer.camera.center = (image.shape[0] / 4, image.shape[1] / 2)
viewer.camera.zoom = 2.0

In [ ]:
# Viewer-level settings: axes
viewer.axes.visible = True

In [ ]:
# Viewer-level settings: grid
viewer.grid.enabled = True
viewer.grid.shape = (3, 4)  # 3 lines x 4 columns grid

In [ ]:
# Viewer-level settings: scale bar
viewer.scale_bar.visible = True
viewer.scale_bar.unit = "um"

## Programmatic layer manipulation

Layers are objects that can be accessed and modified after they are added to the viewer. The layer list is available as `viewer.layers`, which behaves like an ordered mapping. In the next cells, we update layer data, tweak contrast limits, set a scale, and color a labels layer based on object size.

In [ ]:
# Access layers by name or index
layer_names = [layer.name for layer in viewer.layers]
print("These are the layer names:", layer_names)

In [ ]:
# Accessing by name
image_layer = viewer.layers["nuclei"]
labels_layer = viewer.layers["nuclei_labels"]

# Accessing by index
image_layer = viewer.layers[0]
labels_layer = viewer.layers[1]

Like the viewer, layers are Python classes designed to be modified programmatically.

In [ ]:
# Update image data (e.g., apply a gaussian blur)
blurred = filters.gaussian(image, sigma=1, preserve_range=True)
image_layer.data = blurred

In [ ]:
# Adjust contrast limits to highlight details
image_layer.contrast_limits = (np.percentile(blurred, 2), np.percentile(blurred, 98))

In [ ]:
# Set a physical scale (for example, 0.5 µm per pixel)
labels_layer.scale = (0.5, 0.5)
viewer.scale_bar.visible = True
viewer.scale_bar.unit = 'um'

In [ ]:
# Setting the scale of the labels layer back to its default value for the 
# rest of the notebook so that it matches other layers
labels_layer.scale = (1, 1)

Here an example to color labels by area:

In [ ]:
# Color labels by their area using DirectLabelColormap
from napari.utils import DirectLabelColormap

# Compute per-label area with regionprops
regions = measure.regionprops(label_image)
areas = {r.label: r.area for r in regions}
max_area = max(areas.values()) if areas else 1

# Build RGBA colors (blue→red ramp)
color_dict = {0: (0, 0, 0, 0)}
for label_id, area in areas.items():
    t = area / max_area
    color_dict[label_id] = (t, 0.2, 1.0 - t, 1.0)

labels_layer.colormap = color_dict
labels_layer.color_mode = "direct"
labels_layer.opacity = 1

## A simple widget with `magicgui`

Napari can auto-generate GUI widgets from functions using `magicgui`. The key is to **annotate function parameters with Napari types** (e.g., `napari.types.ImageData` or `napari.layers.Image`) and to decorate the function with `@magicgui`. This lets Napari provide layer-aware dropdowns and pass the correct data into your function.

Below, we first define a *plain* watershed segmentation function. It takes arrays and returns arrays. Then we wrap it with a `magicgui` function that uses Napari type annotations and returns `LayerDataTuple` entries so the viewer can create and update layers automatically.

### The "naive" watershed segmentation function

In [ ]:
# Plain function (similar to seg.py): array-in, arrays-out
from scipy.ndimage import gaussian_filter, distance_transform_edt, label
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

def segment_watershed(image, threshold, min_distance, sigma):
    if sigma > 0:
        image = gaussian_filter(image, sigma=sigma)
    mask = image > threshold
    distance = distance_transform_edt(mask)
    local_maxi = peak_local_max(distance, min_distance=min_distance, exclude_border=False)

    markers_array = np.zeros(image.shape, dtype=np.uint8)
    if local_maxi.size:
        markers_array[tuple(local_maxi.T)] = 1

    labels = watershed(-distance, label(markers_array)[0], mask=mask)
    return distance, local_maxi, labels

In [ ]:
# Run the plain function and add its outputs as layers
distance, local_maxi, watershed_labels = segment_watershed(
    image, threshold=int(threshold), min_distance=8, sigma=0.0
)
viewer.add_image(distance, name="distance", colormap="magma")
viewer.add_points(local_maxi, name="seeds", size=4, face_color="cyan")
viewer.add_labels(watershed_labels, name="watershed")

### Widget-based workflow

The key difference between a plain function and a `magicgui` widget is the **type annotations** and the **return type**. With Napari types, the UI knows how to present inputs (e.g., layer dropdowns, sliders) and how to create or update layers from outputs.

In [ ]:
# MagicGUI version: annotated with Napari types (similar to seg_napari.py)
from magicgui import magicgui

# This function with a "@" sign before is called a decorator.
# Here it is used to transform our function into a Qt widget
# that napari can display.
@magicgui(
    auto_call=True, # Setting this option to True automatically reruns the function if a parameter is changed
    threshold={"min": 0, "max": np.max(image), "step": 100},
    min_distance={"min": 1, "max": 40, "step": 1},
    sigma={"min": 0.0, "max": 5.0, "step": 0.1},
)
def watershed_widget(
    image: "napari.types.ImageData",
    threshold: int = np.median(image),
    min_distance: int = 1,
    sigma: float = 0.0,
) -> "napari.types.LayerDataTuple":
    distance, local_maxi, labels = segment_watershed(
        image, threshold, min_distance, sigma
    )
    return [
        (distance, {"name": "distance", "colormap": "magma"}, "Image"),
        (local_maxi, {"name": "seeds", "size": 4, "face_color": "cyan"}, "Points"),
        (labels, {"name": "watershed"}, "Labels"),
    ]

viewer.window.add_dock_widget(watershed_widget, area="right")

## Compare two label layers (agreement + IoU)

The widget below compares a ground‑truth labels layer to a prediction labels layer. It creates an RGBA overlay where **green** indicates agreement (both foreground), **red** indicates disagreement, and background remains transparent. The global IoU is shown beneath the button.

In [ ]:
from magicgui import magicgui
from magicgui.widgets import Label

iou_label = Label(value="IoU: -")

@magicgui(call_button="Compare")
def compare_labels_widget(
    gt: "napari.layers.Labels",
    pred: "napari.layers.Labels",
) -> None:
    gt_fg = gt.data > 0
    pred_fg = pred.data > 0

    intersection = np.logical_and(gt_fg, pred_fg).sum()
    union = np.logical_or(gt_fg, pred_fg).sum()
    iou = intersection / union if union > 0 else 1.0
    iou_label.value = f"IoU: {iou:.3f}"

    agree = np.logical_and(gt_fg, pred_fg)
    disagree = np.logical_xor(gt_fg, pred_fg)

    overlay = np.zeros((*gt_fg.shape, 4), dtype=np.float32)
    overlay[agree] = (0.0, 1.0, 0.0, 1.0)
    overlay[disagree] = (1.0, 0.0, 0.0, 1.0)

    if "label_agreement" in [l.name for l in viewer.layers]:
        existing = viewer.layers["label_agreement"]
        existing.data = overlay
    else:
        viewer.add_image(overlay, name="label_agreement")

        

viewer.window.add_dock_widget(compare_labels_widget, area="right")
viewer.window.add_dock_widget(iou_label, area="right")